## Selecting images and annotation.

50/100/50

U / D / P

In [10]:
import os
import random
import cv2
import pandas as pd
import shutil
import xlsxwriter
from tqdm import tqdm

# ================= CONFIGURATION =================
DATASET_DIR = "/media/holidayj/Documents/Data/Platform/Euljiro/Euljiro_inner_20201128_f1038_t1519/Euljiro_/all_for_FleissKappa"
IMAGE_DIR = os.path.join(DATASET_DIR, "images")
LABEL_DIR = os.path.join(DATASET_DIR, "labels")
OUTPUT_DIR = "./survey_material"
EXCEL_FILENAME = "Annotator_Task_Form.xlsx"

# TARGET DISTRIBUTION (50 / 100 / 50)
TARGET_COUNTS = {
    0: 50,   # Ascending
    1: 100,  # Descending (Main focus)
    2: 50    # Passing
}
# =================================================

def parse_yolo_line(line, img_width, img_height):
    parts = line.strip().split()
    class_id = int(parts[0])
    x_center = float(parts[1]) * img_width
    y_center = float(parts[2]) * img_height
    width = float(parts[3]) * img_width
    height = float(parts[4]) * img_height
    
    x1 = int(x_center - width / 2)
    y1 = int(y_center - height / 2)
    x2 = int(x_center + width / 2)
    y2 = int(y_center + height / 2)
    
    return class_id, x1, y1, x2, y2

def main():
    if os.path.exists(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)
    os.makedirs(OUTPUT_DIR)
    
    # --- 1. Collect All Objects ---
    print("Scanning dataset...")
    all_objects = []
    
    image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.png'))]
    
    for img_file in tqdm(image_files):
        txt_file = img_file.replace('.jpg', '.txt').replace('.png', '.txt')
        txt_path = os.path.join(LABEL_DIR, txt_file)
        img_path = os.path.join(IMAGE_DIR, img_file)
        
        if not os.path.exists(txt_path): continue
            
        img = cv2.imread(img_path)
        if img is None: continue
        h, w, _ = img.shape
        
        with open(txt_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                cls, x1, y1, x2, y2 = parse_yolo_line(line, w, h)
                # Only collect classes 0, 1, 2 (Ignore others if any)
                if cls in TARGET_COUNTS:
                    all_objects.append({
                        'img_name': img_file,
                        'img_path': img_path,
                        'class_id': cls,
                        'box': (x1, y1, x2, y2)
                    })

    # --- 2. Stratified Sampling (50 / 100 / 50) ---
    print(f"Total objects found: {len(all_objects)}")
    
    # Separate lists by class
    objs_0 = [x for x in all_objects if x['class_id'] == 0]
    objs_1 = [x for x in all_objects if x['class_id'] == 1]
    objs_2 = [x for x in all_objects if x['class_id'] == 2]
    
    print(f"Pool size -> Asc: {len(objs_0)}, Desc: {len(objs_1)}, Pass: {len(objs_2)}")
    
    selected_objects = []
    
    # Sample exact amounts (using min to avoid errors if pool is too small)
    selected_objects.extend(random.sample(objs_0, min(len(objs_0), TARGET_COUNTS[0])))
    selected_objects.extend(random.sample(objs_1, min(len(objs_1), TARGET_COUNTS[1])))
    selected_objects.extend(random.sample(objs_2, min(len(objs_2), TARGET_COUNTS[2])))
    
    # Shuffle the final list so annotators don't see all "Ascending" first
    random.shuffle(selected_objects)
    
    print(f"Final Survey Size: {len(selected_objects)} images")
    
    # --- 3. Generate Images & Excel ---
    print("Generating images and Excel sheet...")
    excel_data = []
    ground_truth_data = []
    
    for i, obj in enumerate(tqdm(selected_objects)):
        img = cv2.imread(obj['img_path'])
        x1, y1, x2, y2 = obj['box']
        
        # Draw Blue Box
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        
        out_name = f"survey_{i:03d}.jpg"
        cv2.imwrite(os.path.join(OUTPUT_DIR, out_name), img)
        
        excel_data.append({
            'Image_ID': out_name,
            'Your_Choice': '' 
        })
        
        ground_truth_data.append({
            'Image_ID': out_name,
            'GT_Label': obj['class_id']
        })

    # --- 4. Create Excel with Dropdown (Fixed) ---
    workbook = xlsxwriter.Workbook(EXCEL_FILENAME)
    worksheet = workbook.add_worksheet("AnnotationTask")
    
    header_fmt = workbook.add_format({'bold': True, 'bg_color': '#D3D3D3', 'border': 1})
    
    worksheet.write('A1', 'Image File Name', header_fmt)
    worksheet.write('B1', 'Select Action (Click Cell)', header_fmt)
    worksheet.write('C1', 'Notes (Optional)', header_fmt)
    
    worksheet.set_column('A:A', 20)
    worksheet.set_column('B:B', 25)
    worksheet.set_column('C:C', 30)
    
    for row_idx, item in enumerate(excel_data):
        worksheet.write(row_idx + 1, 0, item['Image_ID'])
        
        # *** FIX: Added correct start_row, start_col, end_row, end_col ***
        worksheet.data_validation(row_idx + 1, 1, row_idx + 1, 1, {
            'validate': 'list',
            'source': ['0: Ascending', '1: Descending', '2: Passing']
        })

    workbook.close()
    
    # Save Key
    pd.DataFrame(ground_truth_data).to_excel("Answer_Key_DO_NOT_SEND.xlsx", index=False)
    
    print("\n[SUCCESS]")
    print(f"Generated {len(selected_objects)} images.")
    print(f"Ascending: {sum(1 for x in selected_objects if x['class_id']==0)}")
    print(f"Descending: {sum(1 for x in selected_objects if x['class_id']==1)}")
    print(f"Passing: {sum(1 for x in selected_objects if x['class_id']==2)}")

if __name__ == "__main__":
    main()

Scanning dataset...


100%|██████████| 6378/6378 [00:08<00:00, 767.58it/s]


Total objects found: 12925
Pool size -> Asc: 4380, Desc: 4236, Pass: 4309
Final Survey Size: 200 images
Generating images and Excel sheet...


100%|██████████| 200/200 [00:00<00:00, 249.27it/s]



[SUCCESS]
Generated 200 images.
Ascending: 50
Descending: 100
Passing: 50


## Analyze the Results

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from statsmodels.stats.inter_rater import fleiss_kappa

# ================= CONFIGURATION =================
# Folder where you saved the returned Excel files from annotators
ANNOTATOR_FILES_DIR = "./returned_files" 

# The Master Key file you generated earlier
GROUND_TRUTH_FILE = "Answer_Key_DO_NOT_SEND.xlsx" 
# =================================================

def clean_label(value):
    """Converts '0: Ascending' or 0 to integer 0"""
    if pd.isna(value):
        return -1 # Missing value
    str_val = str(value).strip()
    if ':' in str_val:
        return int(str_val.split(':')[0])
    try:
        return int(float(str_val))
    except:
        return -1

def main():
    # 1. Load Ground Truth (Optional: strictly for your reference)
    gt_df = pd.read_excel(GROUND_TRUTH_FILE)
    print(f"Loaded Ground Truth: {len(gt_df)} samples")

    # 2. Load Annotator Files
    annotator_files = glob.glob(os.path.join(ANNOTATOR_FILES_DIR, "*.xlsx"))
    print(f"Found {len(annotator_files)} annotator files.")
    
    if len(annotator_files) < 2:
        print("Error: Need at least 2 annotator files to calculate Kappa.")
        return

    # 3. Build the Agreement Matrix
    # We need a DataFrame where columns are Annotators, rows are Images
    df_merged = gt_df[['Image_ID']].copy()
    
    for file_path in annotator_files:
        name = os.path.basename(file_path).replace(".xlsx", "")
        # Read file
        temp_df = pd.read_excel(file_path)
        
        # Extract answers (assuming column B is 'Select Action (Click Cell)')
        # If headers changed, check the index. Usually column 1 (0-index based).
        # We look for the column that contains 'Select Action'
        target_col = [c for c in temp_df.columns if "Select Action" in str(c)][0]
        
        # Clean and Merge
        temp_df[name] = temp_df[target_col].apply(clean_label)
        
        # Merge on Image_ID to ensure alignment
        # (Assumes Image_ID is in Column A)
        id_col = [c for c in temp_df.columns if "Image" in str(c)][0]
        temp_df = temp_df[[id_col, name]]
        
        df_merged = df_merged.merge(temp_df, left_on='Image_ID', right_on=id_col, how='left')
        df_merged.drop(columns=[id_col], inplace=True)

    # 4. Convert to Fleiss' Kappa Format
    # Matrix: Rows = Subjects (Images), Cols = Categories (0, 1, 2)
    # Value = Count of how many raters assigned that category
    
    annotator_cols = [c for c in df_merged.columns if c != 'Image_ID']
    print(f"Annotators included: {annotator_cols}")
    
    # Filter out any rows with missing answers (-1)
    valid_rows = df_merged[annotator_cols].ge(0).all(axis=1)
    if not valid_rows.all():
        print(f"Warning: Dropping {sum(~valid_rows)} images due to missing answers.")
        df_merged = df_merged[valid_rows]

    # Count votes for each category (0, 1, 2)
    # Shape: (N_images, 3_categories)
    N_categories = 3
    fleiss_matrix = np.zeros((len(df_merged), N_categories))
    
    for i, row in df_merged.iterrows():
        votes = row[annotator_cols].values
        for vote in votes:
            fleiss_matrix[i, int(vote)] += 1
            
    # 5. Calculate Kappa
    kappa = fleiss_kappa(fleiss_matrix)
    
    print("-" * 30)
    print(f"FLEISS' KAPPA SCORE: {kappa:.4f}")
    print("-" * 30)
    
    # Interpretation
    if kappa < 0:
        print("Interpretation: Poor agreement (Less than chance)")
    elif 0.01 <= kappa <= 0.20:
        print("Interpretation: Slight agreement")
    elif 0.21 <= kappa <= 0.40:
        print("Interpretation: Fair agreement")
    elif 0.41 <= kappa <= 0.60:
        print("Interpretation: Moderate agreement")
    elif 0.61 <= kappa <= 0.80:
        print("Interpretation: Substantial agreement")
    elif 0.81 <= kappa <= 1.00:
        print("Interpretation: Almost perfect agreement")

if __name__ == "__main__":
    main()

In [9]:
import os
import pandas as pd
import numpy as np
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
import glob

# ================= CONFIGURATION =================
SURVEY_DIR = "/media/holidayj/Documents/github/dup/IAA_survey"
INCLUDE_GROUND_TRUTH = True
CLASS_NAMES = {0: "Ascending", 1: "Descending", 2: "Passing"}
# =================================================

def clean_label(value):
    if pd.isna(value) or str(value).strip() == "": return None
    try: return int(str(value).split(':')[0])
    except ValueError: return None

def calculate_binary_kappa(ratings_matrix, target_class_id):
    """
    Converts the problem to "Target Class" vs "Everything Else"
    and calculates Fleiss' Kappa.
    """
    # Create a binary matrix: 1 if matches target, 0 if not
    binary_matrix = ratings_matrix.applymap(lambda x: 1 if x == target_class_id else 0)
    
    # Aggregate raters (Category 0 vs Category 1)
    agg_data, _ = aggregate_raters(binary_matrix.to_numpy())
    
    # Calculate Kappa
    try:
        kappa = fleiss_kappa(agg_data)
        return kappa
    except:
        return 0.0

def interpret_kappa(k):
    if k < 0: return "Poor"
    if k <= 0.2: return "Slight"
    if k <= 0.4: return "Fair"
    if k <= 0.6: return "Moderate"
    if k <= 0.8: return "Substantial"
    return "Almost Perfect"

def main():
    print(f"Reading files from: {SURVEY_DIR}")
    
    # 1. Load Data (Same as before)
    xlsx_files = glob.glob(os.path.join(SURVEY_DIR, "*.xlsx"))
    annotator_files = [f for f in xlsx_files if not os.path.basename(f).startswith("~$")]
    if not INCLUDE_GROUND_TRUTH:
        annotator_files = [f for f in annotator_files if "Answer_Key" not in f]

    if len(annotator_files) < 2:
        print("Error: Need at least 2 annotator files.")
        return

    merged_df = None
    for i, file_path in enumerate(annotator_files):
        try:
            df = pd.read_excel(file_path).iloc[:, [0, 1]]
            df.columns = ['Image_ID', f'Annotator_{i+1}']
            df[f'Annotator_{i+1}'] = df[f'Annotator_{i+1}'].apply(clean_label)
            
            if merged_df is None: merged_df = df
            else: merged_df = pd.merge(merged_df, df, on='Image_ID', how='inner')
        except: continue

    ratings_matrix = merged_df.drop(columns=['Image_ID']).dropna()
    print(f"Analyzed {len(ratings_matrix)} common images from {len(ratings_matrix.columns)} annotators.\n")

    # 2. Overall Kappa
    agg_data, _ = aggregate_raters(ratings_matrix.to_numpy())
    overall_kappa = fleiss_kappa(agg_data)
    
    print("="*60)
    print(f"OVERALL FLEISS' KAPPA:  {overall_kappa:.4f}  ({interpret_kappa(overall_kappa)})")
    print("="*60)
    print("\nPER-CLASS AGREEMENT (One-vs-Rest):")
    print(f"{'Class Name':<15} | {'Kappa Score':<10} | {'Interpretation'}")
    print("-" * 45)

    # 3. Per-Class Kappa Loop
    for class_id, class_name in CLASS_NAMES.items():
        k = calculate_binary_kappa(ratings_matrix, class_id)
        print(f"{class_name:<15} | {k:.4f}     | {interpret_kappa(k)}")
    
    print("-" * 45)

if __name__ == "__main__":
    main()

Reading files from: /media/holidayj/Documents/github/dup/IAA_survey
Analyzed 200 common images from 4 annotators.

OVERALL FLEISS' KAPPA:  0.7747  (Substantial)

PER-CLASS AGREEMENT (One-vs-Rest):
Class Name      | Kappa Score | Interpretation
---------------------------------------------
Ascending       | 0.8251     | Almost Perfect
Descending      | 0.8243     | Almost Perfect
Passing         | 0.6402     | Substantial
---------------------------------------------


/tmp/ipykernel_60232/3371251548.py:24: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  binary_matrix = ratings_matrix.applymap(lambda x: 1 if x == target_class_id else 0)


## 1. Making survey dataset Version 2. Cropped iamges with full frames. 

In [11]:
import os
import random
import cv2
import glob
import csv
import numpy as np
from tqdm import tqdm
from collections import defaultdict

# ================= CONFIGURATION =================
# 1. Path to Original Video Folder (For Clean Full HD Context)
VIDEO_DIR = '/media/holidayj/Documents/Data/Platform/Euljiro/Euljiro_inner_20201128_f1038_t1519'

# 2. Path to Existing Dataset Images/Labels (The "Cropped" images you mentioned)
DATASET_DIR = '/media/holidayj/Documents/Data/Platform/final_dataset/Euljiro/0_2_Original_dataset_Euljiro_off_peak_inner_20201128_f1038_t1519/3_class/Euljiro_off_peak_TrainVal_balanced_2sec_plus_addition_1sec_FINAL_20260117/TrainVal'

# 3. Output Directory
BASE_OUTPUT_DIR = 'survey_dataset_v3_fixed'

# Survey Configuration
CLASS_MAPPING = {0: 'Ascending', 1: 'Descending', 2: 'Passing'}
TARGET_COUNTS = {'Descending': 100, 'Ascending': 50, 'Passing': 50}
# =================================================

def get_video_file_map(video_dir):
    """Maps Video ID to MP4 path."""
    video_map = {}
    mp4_files = glob.glob(os.path.join(video_dir, "*.mp4"))
    for v_path in mp4_files:
        filename = os.path.basename(v_path)
        try:
            # Assumes format: 1_2020-11-28...
            vid_id = int(filename.split('_')[0])
            video_map[vid_id] = v_path
        except ValueError:
            pass
    return video_map

def parse_labels(dataset_dir):
    """
    Parses labels. Returns a list of all valid objects found.
    """
    objects_by_class = {name: [] for name in CLASS_MAPPING.values()}
    label_files = glob.glob(os.path.join(dataset_dir, "*.txt"))
    
    print(f"Scanning {len(label_files)} label files in dataset folder...")
    
    for l_file in label_files:
        filename_base = os.path.splitext(os.path.basename(l_file))[0]
        
        # Check if corresponding image exists (jpg or png)
        img_path = os.path.join(dataset_dir, filename_base + ".jpg")
        if not os.path.exists(img_path):
            img_path = os.path.join(dataset_dir, filename_base + ".png")
            if not os.path.exists(img_path):
                continue

        # Parse Video ID and Frame Number from filename (e.g., 4_000840)
        try:
            parts = filename_base.split('_')
            vid_id = int(parts[0])
            frame_num = int(parts[1])
        except (ValueError, IndexError):
            continue

        with open(l_file, 'r') as f:
            lines = f.readlines()
            
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_id = int(parts[0])
                if cls_id in CLASS_MAPPING:
                    class_name = CLASS_MAPPING[cls_id]
                    # YOLO format: center_x, center_y, width, height (normalized)
                    bbox = [float(x) for x in parts[1:5]]
                    
                    objects_by_class[class_name].append({
                        'vid_id': vid_id,
                        'frame_num': frame_num,
                        'bbox': bbox,
                        'class_name': class_name,
                        'dataset_img_path': img_path, 
                        'filename_base': filename_base
                    })
    return objects_by_class

def main():
    # Setup Output Paths
    full_dir = os.path.join(BASE_OUTPUT_DIR, 'images_context_hd')
    target_dir = os.path.join(BASE_OUTPUT_DIR, 'images_target_marked')
    os.makedirs(full_dir, exist_ok=True)
    os.makedirs(target_dir, exist_ok=True)
    
    # 1. Map Videos & Parse Dataset Labels
    video_map = get_video_file_map(VIDEO_DIR)
    all_objects = parse_labels(DATASET_DIR)
    
    # 2. Random Sampling
    selected_samples = []
    for class_name, count in TARGET_COUNTS.items():
        available = all_objects[class_name]
        if len(available) < count:
            print(f"Warning: Not enough {class_name} samples. Taking all {len(available)}.")
            selected_samples.extend(available)
        else:
            selected_samples.extend(random.sample(available, count))
    
    # Shuffle for survey
    random.shuffle(selected_samples)
    
    # Assign Survey IDs (1 to 200)
    for i, sample in enumerate(selected_samples):
        sample['survey_id'] = i + 1

    # 3. Group by Video ID for efficient MP4 reading
    samples_by_video = defaultdict(list)
    for sample in selected_samples:
        samples_by_video[sample['vid_id']].append(sample)

    answer_sheet_data = []

    # 4. Processing Loop
    for vid_id, samples in samples_by_video.items():
        
        # Open Video (For Context Image)
        video_available = False
        cap = None
        if vid_id in video_map:
            video_path = video_map[vid_id]
            cap = cv2.VideoCapture(video_path)
            if cap.isOpened():
                video_available = True
            else:
                print(f"Warning: Could not open video {video_path}")
        else:
            print(f"Warning: Video {vid_id} not found. Context image will be skipped.")

        # Sort by frame for efficient seeking
        samples.sort(key=lambda x: x['frame_num'])
        
        for sample in tqdm(samples, desc=f"Video {vid_id}"):
            survey_id = sample['survey_id']
            
            # --- A. Generate Target Image (From DATASET Image - As Is + Box) ---
            ds_img = cv2.imread(sample['dataset_img_path'])
            if ds_img is None:
                continue

            h_ds, w_ds, _ = ds_img.shape
            n_x, n_y, n_w, n_h = sample['bbox']
            
            # Calculate pixel coordinates for the box
            x_center = int(n_x * w_ds)
            y_center = int(n_y * h_ds)
            w_box = int(n_w * w_ds)
            h_box = int(n_h * h_ds)
            
            x1 = int(x_center - w_box / 2)
            y1 = int(y_center - h_box / 2)
            x2 = int(x_center + w_box / 2)
            y2 = int(y_center + h_box / 2)
            
            # Draw Red Box (Thickness 2)
            cv2.rectangle(ds_img, (x1, y1), (x2, y2), (0, 0, 255), 2)
            
            # Save Target Image (No cropping, just the dataset image with box)
            target_name = f"Survey_{survey_id:03d}_Target.jpg"
            cv2.imwrite(os.path.join(target_dir, target_name), ds_img)


            # --- B. Generate Context Image (From FULL HD VIDEO - Clean) ---
            full_name = "N/A"
            if video_available:
                cap.set(cv2.CAP_PROP_POS_FRAMES, sample['frame_num'])
                ret, frame = cap.read()
                if ret:
                    full_name = f"Survey_{survey_id:03d}_Context.jpg"
                    cv2.imwrite(os.path.join(full_dir, full_name), frame)
                else:
                    print(f"Failed to read frame {sample['frame_num']} from video {vid_id}")

            # --- C. Answer Sheet Data ---
            answer_sheet_data.append({
                'Survey_ID': survey_id,
                'Ground_Truth_Class': sample['class_name'],
                'Target_Image': target_name,
                'Context_Image': full_name,
                'Original_Video_ID': vid_id,
                'Frame_Number': sample['frame_num'],
                'Source_Dataset_File': sample['filename_base']
            })
            
        if cap:
            cap.release()

    # 5. Save Answer Sheet
    csv_path = os.path.join(BASE_OUTPUT_DIR, 'answer_sheet.csv')
    with open(csv_path, 'w', newline='') as f:
        fieldnames = ['Survey_ID', 'Ground_Truth_Class', 'Target_Image', 'Context_Image', 
                      'Original_Video_ID', 'Frame_Number', 'Source_Dataset_File']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        answer_sheet_data.sort(key=lambda x: x['Survey_ID'])
        writer.writerows(answer_sheet_data)

    print(f"\nCompleted! Saved {len(answer_sheet_data)} samples to '{BASE_OUTPUT_DIR}'")

if __name__ == "__main__":
    main()

Scanning 5400 label files in dataset folder...


Video 1: 100%|██████████| 19/19 [00:03<00:00,  5.95it/s]


Completed! Saved 200 samples to 'survey_dataset_v3_fixed'


## 2. Making survey tool.

In [30]:
import os
import csv
import json

# ================= CONFIGURATION =================
BASE_OUTPUT_DIR = 'survey_dataset_v3_fixed' 
CSV_PATH = os.path.join(BASE_OUTPUT_DIR, 'answer_sheet.csv')
HTML_OUTPUT = os.path.join(BASE_OUTPUT_DIR, 'survey_tool.html')
# =================================================

def create_html_survey():
    samples = []
    if not os.path.exists(CSV_PATH):
        print(f"Error: {CSV_PATH} not found.")
        return

    with open(CSV_PATH, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            samples.append({
                'id': row['Survey_ID'],
                'target': f"images_target_marked/{row['Target_Image']}",
                'context': f"images_context_hd/{row['Context_Image']}"
            })
    
    samples_json = json.dumps(samples)

    # 이미지 경로
    guide_img_trajectory = "figures/resp3_02_a_annotation_guide.png" 
    guide_img_rush = "figures/resp3_03_b_rush_hour_crop.png"
    guide_img_stair_boundary = "figures/stair_boundary.jpg"
    

    html_content = f"""
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Subway Behavior Annotation (지하철 승객 행동 분류)</title>
    <style>
        body {{ font-family: 'Malgun Gothic', 'Apple SD Gothic Neo', sans-serif; margin: 0; padding: 20px; background-color: #2c3e50; color: #333; }}
        .container {{ max-width: 1800px; margin: 0 auto; background: white; padding: 30px; border-radius: 8px; box-shadow: 0 4px 15px rgba(0,0,0,0.3); }}
        
        /* 가이드라인 스타일 */
        .guideline-section {{ background-color: #f8f9fa; padding: 25px; border-radius: 8px; border: 1px solid #e9ecef; margin-bottom: 30px; }}
        .guideline-title {{ font-size: 1.5em; font-weight: bold; color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; margin-bottom: 15px; }}
        .rule-box {{ margin-bottom: 20px; }}
        .rule-header {{ font-weight: bold; font-size: 1.1em; color: #e74c3c; margin-bottom: 5px; }}
        .rule-content {{ padding-left: 15px; line-height: 1.6; }}
        .rule-content li {{ margin-bottom: 8px; }}
        
        .case-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(300px, 1fr)); gap: 20px; margin-top: 15px; }}
        .case-item {{ background: white; padding: 15px; border: 1px solid #ddd; border-radius: 5px; }}
        .case-item img {{ max-width: 100%; height: auto; border: 1px solid #eee; margin-bottom: 10px; }}
        .case-desc {{ font-size: 0.9em; color: #555; }}
        
        /* C. 제출 안내 스타일 (강조) */
        .submission-guide {{ background-color: #e8f5e9; border-left: 5px solid #28a745; padding: 15px; margin-top: 10px; }}
        
        /* 인터페이스 스타일 */
        .header {{ display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px; border-bottom: 2px solid #eee; padding-bottom: 10px; }}
        h2 {{ margin: 0; color: #2c3e50; }}
        .progress {{ font-size: 1.5em; font-weight: bold; color: #e74c3c; }}
        
        .image-container {{ display: flex; gap: 20px; margin-bottom: 20px; height: 750px; }}
        .box-target {{ flex: 1; display: flex; flex-direction: column; }}
        .box-context {{ flex: 3; display: flex; flex-direction: column; }}
        .img-wrapper {{ flex: 1; border: 2px solid #ccc; background: #000; display: flex; align-items: center; justify-content: center; overflow: hidden; border-radius: 4px; }}
        img {{ max-width: 100%; max-height: 100%; object-fit: contain; }}
        .label {{ font-weight: bold; margin-bottom: 8px; font-size: 1.1em; color: #555; }}
        
        .controls {{ text-align: center; padding: 20px; background: #ecf0f1; border-radius: 8px; }}
        .btn-group {{ display: flex; justify-content: center; gap: 15px; margin-bottom: 15px; flex-wrap: wrap; }}
        button {{ padding: 15px 30px; font-size: 1.1em; border: none; border-radius: 6px; cursor: pointer; transition: 0.2s; box-shadow: 0 2px 5px rgba(0,0,0,0.1); font-family: 'Malgun Gothic', sans-serif; }}
        .btn-choice {{ background-color: #fff; border: 2px solid #bdc3c7; color: #333; font-weight: bold; min-width: 200px; }}
        .btn-choice:hover {{ background-color: #ecf0f1; transform: translateY(-2px); }}
        .btn-choice.selected {{ border-color: #3498db; background-color: #3498db; color: white; }}
        .btn-nav {{ background-color: #34495e; color: white; font-weight: bold; min-width: 120px; }}
        .btn-nav:hover {{ background-color: #2c3e50; }}
        
        .finish-area {{ display: none; margin-top: 20px; padding: 20px; background-color: #d4edda; border: 1px solid #c3e6cb; border-radius: 8px; animation: fadeIn 0.5s; }}
        .finish-msg {{ color: #155724; font-size: 1.2em; margin-bottom: 15px; line-height: 1.5; }}
        .btn-save {{ background-color: #27ae60; color: white; font-size: 1.4em; padding: 15px 50px; }}
        .btn-save:hover {{ background-color: #2ecc71; transform: scale(1.05); }}
        @keyframes fadeIn {{ from {{ opacity: 0; }} to {{ opacity: 1; }} }}
        .instructions {{ text-align: center; color: #7f8c8d; margin-top: 15px; font-size: 1em; }}
    </style>
</head>
<body>

<div class="container">
    
    <div class="guideline-section">
        <div class="guideline-title">Annotation Guideline & Boundary Cases (필독)</div>
        
        <div class="rule-box">
            <div class="rule-header">A. 기본 규칙 (Basic Rules)</div>
            <ul class="rule-content">
                <li><b>1. Ascending (올라감):</b> 계단을 올라가거나, 한 쪽 발(foot)이 보이며 계단 쪽으로 향하는 경우.<br>
                    <span style="color:#d35400; font-weight:bold;">※ 사람에 가린 경우, 가리는 사람이 없다고 가정하고 판단.</span>
                </li>
                <li><b>2. Descending (내려옴/이동):</b> 계단을 내려오거나, 승강장에서 출입문을 향해 이동하는 경우.</li>
                <li><b>3. Passing (그 외):</b> 위 두 경우에 해당하지 않는 모든 경우.</li>
            </ul>
        </div>
        
        <div class="rule-box">
             <div class="rule-header">B. 애매한 경우 판단 기준 (Boundary Cases)</div>
             <div class="case-grid">
                <div class="case-item">
                    <b>1. 방향 모호 (Trajectory)</b><br>
                    <span class="case-desc">문 방향이면 Descending</span>
                    <img src="{guide_img_trajectory}" alt="Guide">
                </div>
                <div class="case-item">
                    <b>2. 줄 서기 (Waiting)</b><br>
                    <span class="case-desc">탑승 대기 줄은 Descending. 파란 박스 모두 Descending 분류.</span>
                    <img src="{guide_img_rush}" alt="Rush">
                </div>
                <div class="case-item">
                    <b>3. 계단 경계 (Stair Boundary)</b><br>
                    <span class="case-desc">승강장 바닥의 노란선(가로방향)까지를 계단으로 본다.</span>
                    <img src="{guide_img_stair_boundary}" alt="Stair Boundary">
                </div>
                
             </div>
        </div>

        <div class="rule-box">
            <div class="rule-header">C. 설문 완료 및 제출 (Submission)</div>
            <div class="submission-guide">
                <b>⚠️ 안내:</b> 총 {len(samples)}개의 설문 문항에 빠짐없이 응답하셔야 합니다.<br>
                모든 문항을 완료하시면 화면 하단에 <b>초록색 '결과 저장' 버튼</b>이 나타납니다.<br>
                버튼을 눌러 <b>.csv 파일을 다운로드</b>한 후, 담당자에게 보내주시기 바랍니다.
            </div>
        </div>
    </div>
    
    <div class="header">
        <h2>Subway Behavior Annotation</h2>
        <div class="progress">Sample <span id="current-index">1</span> / <span id="total-count">0</span></div>
    </div>

    <div class="image-container">
        <div class="box-target">
            <div class="label">대상 확대 (Target)</div>
            <div class="img-wrapper"><img id="img-target" src="" alt="Target"></div>
        </div>
        <div class="box-context">
            <div class="label">전체 상황 (Full HD Context)</div>
            <div class="img-wrapper"><img id="img-context" src="" alt="Context"></div>
        </div>
    </div>

    <div class="controls">
        <div class="btn-group">
            <button class="btn-choice" onclick="selectChoice('Ascending')" id="btn-asc">1. 계단을 올라가거나, 한 쪽 발(foot)이 보이며 계단 쪽으로 향하는 경우<br>(Ascending, 사람에 가린 경우, 가리는 사람이 없다고 가정하고 판단.)</button>
            <button class="btn-choice" onclick="selectChoice('Descending')" id="btn-desc">2. 계단을 내려오거나, 승강장에서 출입문을 향해 이동하는 경우<br>(Descending)</button>
            <button class="btn-choice" onclick="selectChoice('Passing')" id="btn-pass">3. 그 외, 왼쪽 크롭된 이미지만으로 승객의 방향을 판단 불가<br>(Passing)</button>
        </div>
        
        <div class="btn-group">
            <button class="btn-nav" onclick="prevSample()">← 이전</button>
            <button class="btn-nav" onclick="nextSample()">다음 →</button>
        </div>
        
        <div id="finish-area" class="finish-area">
            <div class="finish-msg">
                <b>🎉 모든 설문이 완료되었습니다!</b><br>
                아래 <b>'결과 저장'</b> 버튼을 눌러 파일을 다운로드한 후, 담당자에게 보내주세요.
            </div>
            <button class="btn-save" onclick="downloadCSV()">결과 저장 (Download CSV)</button>
        </div>

        <div class="instructions">단축키: [1], [2], [3] 선택 &nbsp;|&nbsp; [←], [→] 이동</div>
    </div>
</div>

<script>
    const samples = {samples_json};
    let currentIndex = 0;
    const answers = {{}}; 

    document.getElementById('total-count').innerText = samples.length;
    loadSample(0);

    function loadSample(index) {{
        if (index < 0 || index >= samples.length) return;
        currentIndex = index;
        
        const s = samples[index];
        document.getElementById('current-index').innerText = index + 1;
        document.getElementById('img-target').src = s.target;
        document.getElementById('img-context').src = s.context;
        
        updateButtons();
        
        if (Object.keys(answers).length === samples.length) {{
            document.getElementById('finish-area').style.display = 'block';
            document.getElementById('finish-area').scrollIntoView({{behavior: "smooth"}});
        }}
    }}

    function selectChoice(choice) {{
        const id = samples[currentIndex].id;
        answers[id] = choice;
        updateButtons();
        if (currentIndex < samples.length - 1) {{
            setTimeout(() => nextSample(), 150); 
        }} else {{
            loadSample(currentIndex);
        }}
    }}

    function updateButtons() {{
        const id = samples[currentIndex].id;
        const currentAnswer = answers[id];
        ['Ascending', 'Descending', 'Passing'].forEach(opt => {{
            const btn = document.getElementById('btn-' + (opt === 'Ascending' ? 'asc' : opt === 'Descending' ? 'desc' : 'pass'));
            if (opt === currentAnswer) btn.classList.add('selected');
            else btn.classList.remove('selected');
        }});
    }}

    function nextSample() {{ loadSample(currentIndex + 1); }}
    function prevSample() {{ loadSample(currentIndex - 1); }}

    function downloadCSV() {{
        let csvContent = "data:text/csv;charset=utf-8,Survey_ID,Selected_Class\\n";
        const sortedIds = Object.keys(answers).sort((a,b) => parseInt(a) - parseInt(b));
        sortedIds.forEach(id => {{
            csvContent += id + "," + answers[id] + "\\n";
        }});
        const encodedUri = encodeURI(csvContent);
        const link = document.createElement("a");
        link.setAttribute("href", encodedUri);
        link.setAttribute("download", "survey_results.csv");
        document.body.appendChild(link);
        link.click();
    }}

    document.addEventListener('keydown', function(event) {{
        if (event.key === '1') selectChoice('Ascending');
        if (event.key === '2') selectChoice('Descending');
        if (event.key === '3') selectChoice('Passing');
        if (event.key === 'ArrowRight') nextSample();
        if (event.key === 'ArrowLeft') prevSample();
    }});
</script>
</body>
</html>
    """

    with open(HTML_OUTPUT, 'w', encoding='utf-8') as f:
        f.write(html_content)

    print(f"Updated: {HTML_OUTPUT}")
    print("설문 완료 및 제출 안내(C 섹션)가 추가되었습니다.")

if __name__ == "__main__":
    create_html_survey()

Updated: survey_dataset_v3_fixed/survey_tool.html
설문 완료 및 제출 안내(C 섹션)가 추가되었습니다.


## 3. Comparing to answer sheet.

In [29]:
import pandas as pd
from sklearn.metrics import accuracy_score, cohen_kappa_score, classification_report, confusion_matrix

# 1. 파일 경로 설정 (경로를 본인 환경에 맞게 수정하세요)
answer_file = 'survey_dataset_v3_fixed/answer_sheet.csv'  # 정답지
result_file = '/home/holidayj/Downloads/survey_results (1).csv'                        # 설문 결과 (방금 업로드하신 파일)

# 2. 데이터 로드
try:
    df_ans = pd.read_csv(answer_file)
    df_res = pd.read_csv(result_file)
    
    # Survey_ID 기준으로 병합 (순서가 섞여 있어도 ID로 매칭)
    merged = pd.merge(df_ans, df_res, on='Survey_ID', how='inner')
    
    print(f"총 {len(merged)}개의 샘플이 매칭되었습니다.\n")

    # 3. 성능 평가
    y_true = merged['Ground_Truth_Class'] # 정답
    y_pred = merged['Selected_Class']     # 설문 응답

    # 정확도 및 카파 계수
    acc = accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)

    print(f"=== 평가 결과 ===")
    print(f"정확도 (Accuracy): {acc:.2%} ({acc:.4f})")
    print(f"카파 계수 (Cohen's Kappa): {kappa:.4f}")
    
    # 4. 클래스별 상세 리포트
    print("\n=== 상세 리포트 ===")
    print(classification_report(y_true, y_pred))

    # 5. 혼동 행렬 (Confusion Matrix)
    print("=== 혼동 행렬 (Confusion Matrix) ===")
    labels = ['Ascending', 'Descending', 'Passing']
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    df_cm = pd.DataFrame(cm, index=[f"True {x}" for x in labels], columns=[f"Pred {x}" for x in labels])
    print(df_cm)

except FileNotFoundError:
    print("파일을 찾을 수 없습니다. 경로를 확인해 주세요.")
except Exception as e:
    print(f"오류 발생: {e}")

총 200개의 샘플이 매칭되었습니다.

=== 평가 결과 ===
정확도 (Accuracy): 96.00% (0.9600)
카파 계수 (Cohen's Kappa): 0.9364

=== 상세 리포트 ===
              precision    recall  f1-score   support

   Ascending       0.98      0.98      0.98        50
  Descending       0.98      0.95      0.96       100
     Passing       0.91      0.96      0.93        50

    accuracy                           0.96       200
   macro avg       0.96      0.96      0.96       200
weighted avg       0.96      0.96      0.96       200

=== 혼동 행렬 (Confusion Matrix) ===
                 Pred Ascending  Pred Descending  Pred Passing
True Ascending               49                0             1
True Descending               1               95             4
True Passing                  0                2            48


## 1. Frame extractor.
Random extraction from TrainVal set and select one object and extract -30 -20 -10 frames from the video. So that we can make the survey materials.

In [24]:
import os
import random
import cv2
import glob
import csv
import json
import numpy as np
from tqdm import tqdm
from collections import defaultdict

# ================= CONFIGURATION =================
# 1. Video Directory
VIDEO_DIR = '/media/holidayj/Documents/Data/Platform/Euljiro/Euljiro_inner_20201128_f1038_t1519'

# 2. Label Directory
DATASET_DIR = '/media/holidayj/Documents/Data/Platform/final_dataset/Euljiro/0_2_Original_dataset_Euljiro_off_peak_inner_20201128_f1038_t1519/3_class/Euljiro_off_peak_Test'

# 3. Output Directory
BASE_OUTPUT_DIR = 'survey_dataset_final_v6_extended'

# 4. FIXED CROP COORDINATES (320x320)
FIXED_X1 = 1327
FIXED_X2 = 1647
FIXED_Y1 = 120
FIXED_Y2 = 440
CROP_W = FIXED_X2 - FIXED_X1
CROP_H = FIXED_Y2 - FIXED_Y1

# Settings
CLASS_MAPPING = {0: 'Ascending', 1: 'Descending', 2: 'Passing'}
TARGET_COUNTS = {'Descending': 100, 'Ascending': 50, 'Passing': 50}
# =================================================

def get_video_file_map(video_dir):
    video_map = {}
    mp4_files = glob.glob(os.path.join(video_dir, "*.mp4"))
    for v_path in mp4_files:
        filename = os.path.basename(v_path)
        try:
            vid_id = int(filename.split('_')[0])
            video_map[vid_id] = v_path
        except ValueError:
            pass
    return video_map

def parse_labels(dataset_dir):
    objects_by_class = {name: [] for name in CLASS_MAPPING.values()}
    label_files = glob.glob(os.path.join(dataset_dir, "*.txt"))
    print(f"Scanning {len(label_files)} label files...")
    
    for l_file in label_files:
        filename_base = os.path.splitext(os.path.basename(l_file))[0]
        img_path = os.path.join(dataset_dir, filename_base + ".jpg")
        if not os.path.exists(img_path):
            img_path = os.path.join(dataset_dir, filename_base + ".png")
            if not os.path.exists(img_path): continue

        try:
            parts = filename_base.split('_')
            vid_id = int(parts[0])
            frame_num = int(parts[1])
        except (ValueError, IndexError):
            continue

        with open(l_file, 'r') as f:
            lines = f.readlines()
        
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_id = int(parts[0])
                if cls_id in CLASS_MAPPING:
                    class_name = CLASS_MAPPING[cls_id]
                    bbox = [float(x) for x in parts[1:5]]
                    objects_by_class[class_name].append({
                        'vid_id': vid_id,
                        'frame_num': frame_num,
                        'bbox': bbox,
                        'class_name': class_name
                    })
    return objects_by_class

def main():
    # Create Directories
    img_out_dir = os.path.join(BASE_OUTPUT_DIR, 'images_generated')
    os.makedirs(img_out_dir, exist_ok=True)
    
    video_map = get_video_file_map(VIDEO_DIR)
    all_objects = parse_labels(DATASET_DIR)
    
    # Sampling
    selected_samples = []
    for class_name, count in TARGET_COUNTS.items():
        available = all_objects[class_name]
        if len(available) < count:
            selected_samples.extend(available)
        else:
            selected_samples.extend(random.sample(available, count))
    
    random.shuffle(selected_samples)
    for i, sample in enumerate(selected_samples):
        sample['survey_id'] = i + 1

    samples_by_video = defaultdict(list)
    for sample in selected_samples:
        samples_by_video[sample['vid_id']].append(sample)

    answer_sheet_data = []

    # Processing Loop
    for vid_id, samples in samples_by_video.items():
        if vid_id not in video_map:
            print(f"Skipping Video {vid_id}")
            continue
            
        cap = cv2.VideoCapture(video_map[vid_id])
        if not cap.isOpened(): continue

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        v_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        v_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        samples.sort(key=lambda x: x['frame_num'])
        
        for sample in tqdm(samples, desc=f"Video {vid_id}"):
            survey_id = sample['survey_id']
            center_frame_num = sample['frame_num']
            n_x, n_y, n_w, n_h = sample['bbox'] 
            
            # --- 1. Extract Crops (-30, -20, -10, 0) ---
            offsets_crop = [-30, -20, -10, 0]
            filenames = {}
            
            for offset in offsets_crop:
                target_f = center_frame_num + offset
                
                # Determine filenames
                if offset == 0: 
                    suffix = "target_crop"
                    suffix_orig = "target_original"
                else: 
                    suffix = f"minus{abs(offset)}_crop"
                
                fname = f"S{survey_id:03d}_{suffix}.jpg"
                save_path = os.path.join(img_out_dir, fname)
                filenames[suffix] = fname
                
                # For t=0, also prepare original path
                if offset == 0:
                    fname_orig = f"S{survey_id:03d}_{suffix_orig}.jpg"
                    save_path_orig = os.path.join(img_out_dir, fname_orig)
                    filenames[suffix_orig] = fname_orig

                if 0 <= target_f < total_frames:
                    cap.set(cv2.CAP_PROP_POS_FRAMES, target_f)
                    ret, frame = cap.read()
                    if ret:
                        # Fixed Crop
                        y1, y2 = max(0, FIXED_Y1), min(v_height, FIXED_Y2)
                        x1, x2 = max(0, FIXED_X1), min(v_width, FIXED_X2)
                        crop = frame[y1:y2, x1:x2].copy()
                        
                        # Special Handling for Target (t=0)
                        if offset == 0:
                            # 1. Save Original (Raw) Crop first
                            cv2.imwrite(save_path_orig, crop)
                            
                            # 2. Draw BLUE Box on the crop copy
                            # (OpenCV uses BGR: Blue is (255, 0, 0))
                            cx, cy = n_x * CROP_W, n_y * CROP_H
                            bw, bh = n_w * CROP_W, n_h * CROP_H
                            bx1, by1 = int(cx - bw/2), int(cy - bh/2)
                            bx2, by2 = int(cx + bw/2), int(cy + bh/2)
                            
                            cv2.rectangle(crop, (bx1, by1), (bx2, by2), (255, 0, 0), 3) # Blue
                            cv2.imwrite(save_path, crop)
                        else:
                            # For past frames, just save the crop
                            cv2.imwrite(save_path, crop)
                    else:
                        # Handle read failure (black image)
                        black_img = np.zeros((CROP_H, CROP_W, 3), np.uint8)
                        cv2.imwrite(save_path, black_img)
                        if offset == 0: cv2.imwrite(save_path_orig, black_img)
                else:
                    # Handle out of bounds
                    black_img = np.zeros((CROP_H, CROP_W, 3), np.uint8)
                    cv2.imwrite(save_path, black_img)
                    if offset == 0: cv2.imwrite(save_path_orig, black_img)

            # --- 2. Extract Full Frame (t=0) ---
            fname_full = f"S{survey_id:03d}_full_context.jpg"
            save_path_full = os.path.join(img_out_dir, fname_full)
            filenames['full'] = fname_full
            
            if 0 <= center_frame_num < total_frames:
                cap.set(cv2.CAP_PROP_POS_FRAMES, center_frame_num)
                ret, frame = cap.read()
                if ret:
                    cv2.imwrite(save_path_full, frame)
                else:
                    cv2.imwrite(save_path_full, np.zeros((v_height, v_width, 3), np.uint8))
            else:
                cv2.imwrite(save_path_full, np.zeros((v_height, v_width, 3), np.uint8))

            # Record Data (Added Img_Target_Original)
            answer_sheet_data.append({
                'Survey_ID': survey_id,
                'Ground_Truth_Class': sample['class_name'],
                'Img_Minus30': filenames['minus30_crop'],
                'Img_Minus20': filenames['minus20_crop'],
                'Img_Minus10': filenames['minus10_crop'],
                'Img_Target': filenames['target_crop'],          # Has Blue Box
                'Img_Target_Original': filenames['target_original'], # Raw (No Box)
                'Img_Full': filenames['full']
            })
            
        cap.release()

    # Save CSV
    csv_path = os.path.join(BASE_OUTPUT_DIR, 'answer_sheet.csv')
    with open(csv_path, 'w', newline='') as f:
        # Added 'Img_Target_Original' to header
        header = ['Survey_ID', 'Ground_Truth_Class', 'Img_Minus30', 'Img_Minus20', 
                  'Img_Minus10', 'Img_Target', 'Img_Target_Original', 'Img_Full']
        writer = csv.DictWriter(f, fieldnames=header)
        writer.writeheader()
        answer_sheet_data.sort(key=lambda x: x['Survey_ID'])
        writer.writerows(answer_sheet_data)

    print(f"\n[Done] Generated {len(answer_sheet_data)} samples.")
    # Note: create_html_tool call removed as it's separate script, 
    # but the data structure is ready for it.

if __name__ == "__main__":
    main()

Scanning 536 label files...


Video 4: 100%|██████████| 200/200 [02:10<00:00,  1.53it/s]


[Done] Generated 200 samples.


## Writing html survey tool.

In [ ]:
import os
import csv
import json

# ================= CONFIGURATION =================
BASE_OUTPUT_DIR = 'survey_dataset_final_v6_extended' 

CSV_PATH = os.path.join(BASE_OUTPUT_DIR, 'answer_sheet.csv')
HTML_OUTPUT = os.path.join(BASE_OUTPUT_DIR, 'survey_tool.html')
# =================================================

def create_html_tool():
    if not os.path.exists(CSV_PATH):
        print(f"Error: {CSV_PATH} not found.")
        return

    # 1. Read CSV Data
    data = []
    with open(CSV_PATH, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            data.append(row)
    
    # 2. Prepare JSON 
    samples = []
    for row in data:
        samples.append({
            'id': row['Survey_ID'],
            'm30': f"images_generated/{row['Img_Minus30']}",
            'm20': f"images_generated/{row['Img_Minus20']}",
            'm10': f"images_generated/{row['Img_Minus10']}",
            'tgt': f"images_generated/{row['Img_Target']}", # Blue Bounding Box Version
            'full': f"images_generated/{row['Img_Full']}"
        })
    samples_json = json.dumps(samples)
    
    # 3. Image Paths for Guidelines
    guide_img_trajectory = "figures/resp3_02_a_annotation_guide.png" 
    guide_img_rush = "figures/resp3_03_b_rush_hour_crop.png"
    guide_img_barrier = "figures/stair_boundary.jpg"

    # 4. HTML Content (Updated with your specific text)
    html_content = f"""
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>Subway Behavior Annotation Tool</title>
    <style>
        body {{ font-family: 'Malgun Gothic', 'Apple SD Gothic Neo', sans-serif; background: #2c3e50; padding: 20px; color: #333; }}
        .container {{ max-width: 1800px; margin: 0 auto; background: white; padding: 30px; border-radius: 8px; box-shadow: 0 4px 15px rgba(0,0,0,0.3); }}
        
        /* Guideline Section */
        .guideline-section {{ background: #f8f9fa; padding: 20px; margin-bottom: 20px; border-radius: 8px; border: 1px solid #ddd; }}
        .rule-header {{ font-weight: bold; color: #2c3e50; font-size: 1.2em; border-bottom: 2px solid #3498db; padding-bottom: 10px; margin-bottom: 10px; }}
        .rule-list li {{ margin-bottom: 8px; font-size: 1.05em; }}
        
        /* Layout */
        .display-area {{ display: flex; flex-direction: column; gap: 20px; margin-bottom: 20px; }}
        
        /* Top Row: Full Frame */
        .full-row {{ text-align: center; }}
        .full-box img {{ max-width: 100%; height: auto; max-height: 550px; border: 2px solid #333; border-radius: 4px; }}
        .full-label {{ font-weight: bold; margin-bottom: 5px; color: #333; font-size: 1.1em; }}

        /* Bottom Row: 4 Crops */
        .crops-row {{ display: flex; gap: 15px; justify-content: center; }}
        .crop-box {{ text-align: center; }}
        .crop-box img {{ width: 280px; height: 280px; border: 1px solid #ccc; object-fit: contain; border-radius: 4px; background: #eee; }}
        .crop-label {{ font-weight: bold; margin-bottom: 5px; color: #555; }}
        
        /* Target Box Styles */
        .target-box img {{ border: 2px solid #3498db; }}
        .target-label {{ color: #2980b9; font-size: 1.1em; }}

        /* Controls */
        .controls {{ text-align: center; background: #ecf0f1; padding: 20px; border-radius: 8px; }}
        .btn-group {{ display: flex; flex-direction: column; align-items: center; gap: 10px; margin-bottom: 20px; }}
        
        .btn-choice {{ 
            width: 80%; max-width: 900px; padding: 15px; background-color: #fff; 
            border: 2px solid #bdc3c7; color: #333; font-weight: bold; font-size: 1.0em; 
            border-radius: 6px; cursor: pointer; text-align: left; line-height: 1.4;
        }}
        .btn-choice:hover {{ background-color: #ecf0f1; transform: translateY(-2px); }}
        .btn-choice.selected {{ border-color: #3498db; background-color: #3498db; color: white; }}
        
        .btn-nav {{ padding: 10px 30px; background: #34495e; color: white; border: none; cursor: pointer; font-weight: bold; font-size: 1.1em; border-radius: 5px; margin: 0 10px; }}
        .btn-nav:hover {{ background: #2d3436; }}
        
        .finish-area {{ display: none; margin-top: 20px; padding: 20px; background: #d4edda; border: 1px solid #c3e6cb; text-align: center; border-radius: 8px; }}
        .finish-msg {{ color: #155724; font-size: 1.2em; margin-bottom: 10px; font-weight: bold; }}
        .btn-save {{ background: #27ae60; color: white; padding: 15px 50px; font-size: 1.3em; border: none; cursor: pointer; border-radius: 6px; }}
        .btn-save:hover {{ background: #2ecc71; }}
        
        .case-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(250px, 1fr)); gap: 15px; margin-top: 15px; }}
        .case-item {{ background: white; padding: 10px; border: 1px solid #ddd; border-radius: 5px; }}
        .case-item img {{ max-width: 100%; height: auto; border: 1px solid #eee; margin-top: 5px; }}
    </style>
</head>
<body>
<div class="container">
    
    <div class="guideline-section">
        <div class="rule-header">Annotation Guideline & Boundary Cases (필독)</div>
        <div style="margin-bottom: 20px;">
            <b>A. 기본 규칙 (Basic Rules)</b>
            <ul class="rule-list">
                <li><b>1. Ascending (올라감):</b> 계단을 올라가거나, 한 쪽 발(foot)이 보이며 계단 쪽으로 향하는 경우.<br>
                    <span style="color:#d35400; font-weight:bold;">※ 사람에 가린 경우, 가리는 사람이 없다고 가정하고 판단.</span>
                </li>
                <li><b>2. Descending (내려옴/이동):</b> 계단을 내려오거나, 승강장에서 출입문을 향해 이동하는 경우.</li>
                <li><b>3. Passing (그 외):</b> 위 두 경우에 해당하지 않는 모든 경우.</li>
            </ul>
        </div>
        
        <div class="case-grid">
            <div class="case-item"><b>방향 모호 (Trajectory)</b><br><span style="font-size:0.9em">계단을 내려와 출입문 방향이면 Descending</span><br><img src="{guide_img_trajectory}" alt="Guide"></div>
            <div class="case-item"><b>줄 서기 (Waiting)</b><br><span style="font-size:0.9em">탑승 대기 줄은 Descending. 두번째 frame의 파란 박스는 모두 Descending</span><br><img src="{guide_img_rush}" alt="Rush"></div>
            <div class="case-item"><b>계단 경계 (Stair Boundary)(Excluded)</b><br><span style="font-size:0.9em">승강장 바닥의 노란선(가로방향)까지를 계단으로 본다. (아래 승객은 아직 계단에 있다고 봄.)</span><br><img src="{guide_img_barrier}" alt="Barrier"></div>
        </div>

        <div style="margin-top:15px; background-color:#e8f5e9; border-left:5px solid #28a745; padding:15px;">
            <b>⚠️ 안내:</b> 총 {len(data)}문항을 모두 완료하면 하단에 <b>'결과 저장'</b> 버튼이 나타납니다.
        </div>
    </div>

    <div style="text-align:center; margin-bottom:10px; font-size:1.2em;">
        <h2>Sample <span id="current-index">1</span> / <span id="total-count">0</span></h2>
    </div>

    <div class="display-area">
        <div class="full-row">
            <div class="full-label">Full HD Context (t)</div>
            <div class="full-box">
                <img id="img-full" src="">
            </div>
        </div>

        <div class="crops-row">
            <div class="crop-box"><div class="crop-label">t - 30</div><img id="img-m30" src=""></div>
            <div class="crop-box"><div class="crop-label">t - 20</div><img id="img-m20" src=""></div>
            <div class="crop-box"><div class="crop-label">t - 10</div><img id="img-m10" src=""></div>
            <div class="crop-box target-box"><div class="crop-label target-label">Target (t)</div><img id="img-tgt" src=""></div>
        </div>
    </div>

    <div class="controls">
        <div class="btn-group">
            <button class="btn-choice" onclick="selectChoice('Ascending')" id="btn-asc">1. 계단을 올라가거나, 한 쪽 발(foot)이 보이며 계단 쪽으로 향하는 경우<br>(Ascending, 사람에 가린 경우 가정하여 판단)</button>
            <button class="btn-choice" onclick="selectChoice('Descending')" id="btn-desc">2. 계단을 내려오거나, 승강장에서 출입문을 향해 이동하는 경우<br>(Descending)</button>
            <button class="btn-choice" onclick="selectChoice('Passing')" id="btn-pass">3. 그 외 (판단 불가 경우 포함)<br>(Passing)</button>
        </div>
        
        <div style="margin-top:10px;">
            <button class="btn-nav" onclick="prevSample()">← 이전 (Left)</button>
            <button class="btn-nav" onclick="nextSample()">다음 (Right) →</button>
        </div>

        <div id="finish-area" class="finish-area">
            <div class="finish-msg">🎉 모든 설문이 완료되었습니다!</div>
            <button class="btn-save" onclick="downloadCSV()">결과 저장 (Download CSV)</button>
        </div>
        <div style="color:#7f8c8d; margin-top:10px;">단축키: [1], [2], [3] 선택 &nbsp;|&nbsp; [←], [→] 이동</div>
    </div>
</div>

<script>
    const samples = {samples_json};
    let currentIndex = 0;
    const answers = {{}}; 
    
    document.getElementById('total-count').innerText = samples.length;
    loadSample(0);

    function loadSample(index) {{
        if (index < 0 || index >= samples.length) return;
        currentIndex = index;
        const s = samples[index];
        
        document.getElementById('current-index').innerText = index + 1;
        document.getElementById('img-m30').src = s.m30;
        document.getElementById('img-m20').src = s.m20;
        document.getElementById('img-m10').src = s.m10;
        document.getElementById('img-tgt').src = s.tgt;
        document.getElementById('img-full').src = s.full;
        
        updateButtons();
        checkCompletion();
    }}

    function selectChoice(c) {{
        answers[samples[currentIndex].id] = c;
        updateButtons();
        if (currentIndex < samples.length - 1) setTimeout(() => loadSample(currentIndex + 1), 150);
        else checkCompletion();
    }}

    function checkCompletion() {{
        if (Object.keys(answers).length === samples.length) {{
            document.getElementById('finish-area').style.display = 'block';
            document.getElementById('finish-area').scrollIntoView({{behavior: "smooth"}});
        }}
    }}

    function updateButtons() {{
        const ans = answers[samples[currentIndex].id];
        const mapping = {{'Ascending': 'asc', 'Descending': 'desc', 'Passing': 'pass'}};
        Object.keys(mapping).forEach(opt => {{
            const btn = document.getElementById('btn-' + mapping[opt]);
            if (opt === ans) btn.classList.add('selected');
            else btn.classList.remove('selected');
        }});
    }}
    
    function nextSample() {{ loadSample(currentIndex + 1); }}
    function prevSample() {{ loadSample(currentIndex - 1); }}
    
    function downloadCSV() {{
        let csv = "data:text/csv;charset=utf-8,Survey_ID,Selected_Class\\n";
        // Sort by ID naturally
        const sortedIds = Object.keys(answers).sort((a,b) => parseInt(a)-parseInt(b));
        sortedIds.forEach(id => csv += id + "," + answers[id] + "\\n");
        const link = document.createElement("a");
        link.href = encodeURI(csv);
        link.download = "annotation_results_final.csv";
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
    }}
    
    document.addEventListener('keydown', e => {{
        if(e.key==='1') selectChoice('Ascending');
        if(e.key==='2') selectChoice('Descending');
        if(e.key==='3') selectChoice('Passing');
        if(e.key==='ArrowRight') nextSample();
        if(e.key==='ArrowLeft') prevSample();
    }});
</script>
</body>
</html>
    """
    
    with open(HTML_OUTPUT, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"Success: Tool generated at {HTML_OUTPUT}")

if __name__ == "__main__":
    create_html_tool()

Success: Tool generated at survey_dataset_final_v6_extended/survey_tool.html


In [31]:
import os
import csv
import json

# ================= CONFIGURATION =================
BASE_OUTPUT_DIR = 'survey_dataset_final_v6_extended' 

CSV_PATH = os.path.join(BASE_OUTPUT_DIR, 'answer_sheet.csv')
HTML_OUTPUT = os.path.join(BASE_OUTPUT_DIR, 'survey_tool.html')
# =================================================

def create_html_tool():
    if not os.path.exists(CSV_PATH):
        print(f"Error: {CSV_PATH} not found.")
        return

    # 1. Read CSV Data
    data = []
    with open(CSV_PATH, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            data.append(row)
    
    # 2. Prepare JSON 
    # Logic: Using 'Img_Target' (Blue Box Version) as per previous context
    samples = []
    for row in data:
        samples.append({
            'id': row['Survey_ID'],
            'm30': f"images_generated/{row['Img_Minus30']}",
            'm20': f"images_generated/{row['Img_Minus20']}",
            'm10': f"images_generated/{row['Img_Minus10']}",
            'tgt': f"images_generated/{row['Img_Target']}", 
            'full': f"images_generated/{row['Img_Full']}"
        })
    samples_json = json.dumps(samples)
    
    # 3. Image Paths for Guidelines
    guide_img_trajectory = "figures/resp3_02_a_annotation_guide.png" 
    guide_img_rush = "figures/resp3_03_b_rush_hour_crop.png"
    guide_img_barrier = "figures/stair_boundary.jpg"

    # 4. HTML Content (Exact Text & Layout from uploaded file)
    html_content = f"""
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>Subway Behavior Annotation</title>
    <style>
        body {{ font-family: 'Malgun Gothic', 'Apple SD Gothic Neo', sans-serif; background: #2c3e50; padding: 20px; }}
        .container {{ max-width: 1800px; margin: 0 auto; background: white; padding: 30px; border-radius: 8px; }}

        /* Guideline Section */
        .guideline-section {{ background: #f8f9fa; padding: 20px; margin-bottom: 20px; border-radius: 8px; border: 1px solid #ddd; }}
        .rule-header {{ font-weight: bold; color: #2c3e50; font-size: 1.2em; border-bottom: 2px solid #3498db; padding-bottom: 10px; margin-bottom: 10px; }}
        .rule-list li {{ margin-bottom: 8px; font-size: 1.05em; }}

        /* Layout */
        .display-area {{ display: flex; flex-direction: column; gap: 20px; margin-bottom: 20px; }}

        /* Top Row: Full Frame */
        .full-row {{ text-align: center; }}
        .full-box img {{ max-width: 100%; height: auto; max-height: 550px; border: 2px solid #333; }}
        .full-label {{ font-weight: bold; margin-bottom: 5px; color: #333; font-size: 1.1em; }}

        /* Bottom Row: 4 Crops */
        .crops-row {{ display: flex; gap: 15px; justify-content: center; }}
        .crop-box {{ text-align: center; }}
        .crop-box img {{ width: 280px; height: 280px; border: 1px solid #ccc; object-fit: contain; }}
        .crop-label {{ font-weight: bold; margin-bottom: 5px; color: #555; }}

        /* Target Box: Red border removed per user request as objects are already boxed */
        .target-box img {{ border: 1px solid #ccc; }} 
        .target-label {{ color: #333; font-size: 1.1em; }}

        /* Controls & Buttons */
        .controls {{ text-align: center; background: #ecf0f1; padding: 20px; border-radius: 8px; }}

        .btn-group {{ display: flex; flex-direction: column; align-items: center; gap: 10px; margin-bottom: 20px; }}

        .btn-choice {{ 
            width: 80%; 
            max-width: 900px; 
            padding: 15px; 
            background-color: #fff; 
            border: 2px solid #bdc3c7; 
            color: #333; 
            font-weight: bold; 
            font-size: 1.0em; 
            border-radius: 6px; 
            cursor: pointer; 
            text-align: left;
            line-height: 1.4;
        }}
        .btn-choice:hover {{ background-color: #ecf0f1; transform: translateY(-2px); }}
        .btn-choice.selected {{ border-color: #3498db; background-color: #3498db; color: white; }}

        .btn-nav {{ padding: 10px 30px; background: #34495e; color: white; border: none; cursor: pointer; font-weight: bold; font-size: 1.1em; border-radius: 5px; margin: 0 10px; }}
        .btn-nav:hover {{ background: #2d3436; }}

        .finish-area {{ display: none; margin-top: 20px; padding: 20px; background: #d4edda; border: 1px solid #c3e6cb; text-align: center; border-radius: 8px; }}
        .finish-msg {{ color: #155724; font-size: 1.2em; margin-bottom: 10px; font-weight: bold; }}
        .btn-save {{ background: #27ae60; color: white; padding: 15px 50px; font-size: 1.3em; border: none; cursor: pointer; border-radius: 6px; }}
        .btn-save:hover {{ background: #2ecc71; }}

        .case-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(250px, 1fr)); gap: 15px; margin-top: 15px; }}
        .case-item {{ background: white; padding: 10px; border: 1px solid #ddd; border-radius: 5px; }}
        .case-item img {{ max-width: 100%; height: auto; border: 1px solid #eee; }}
    </style>
</head>
<body>
<div class="container">

    <div class="guideline-section">
        <div class="rule-header">Annotation Guideline & Boundary Cases (필독)</div>
        <div style="margin-bottom: 20px;">
            <b>A. 기본 규칙 (Basic Rules)</b>
            <ul class="rule-list">
                <li><b>1. Ascending (올라감):</b> 계단을 올라가거나, 한 쪽 발(foot)이 보이며 계단 쪽으로 향하는 경우.<br>
                    <span style="color:#d35400; font-weight:bold;">※ 사람에 가린 경우, 가리는 사람이 없다고 가정하고 판단.</span>
                </li>
                <li><b>2. Descending (내려옴/이동):</b> 계단을 내려오거나, 승강장에서 출입문을 향해 이동하는 경우.</li>
                <li><b>3. Passing (그 외):</b> 위 두 경우에 해당하지 않는 모든 경우.</li>
            </ul>
        </div>

        <div class="case-grid">
            <div class="case-item"><b>방향 모호 (Trajectory)</b><br><span style="font-size:0.9em">계단을 내려와 출입문 방향이면 Descending</span><br><img src="{guide_img_trajectory}" alt="Guide"></div>
            <div class="case-item"><b>줄 서기 (Waiting)</b><br><span style="font-size:0.9em">탑승 대기 줄은 Descending. 두번째 frame의 파란 박스는 모두 Descending</span><br><img src="{guide_img_rush}" alt="Rush"></div>
            <div class="case-item"><b>계단 경계 (Stair Boundary)(Excluded)</b><br><span style="font-size:0.9em">승강장 바닥의 노란선(가로방향)까지를 계단으로 본다. (아래 승객은 아직 계단에 있다고 봄.)</span><br><img src="{guide_img_barrier}" alt="Barrier"></div>
        </div>

        <div style="margin-top:15px; background-color:#e8f5e9; border-left:5px solid #28a745; padding:15px;">
            <b>⚠️ 안내:</b> 총 {len(data)}문항을 모두 완료하면 하단에 <b>'결과 저장'</b> 버튼이 나타납니다.
        </div>
    </div>

    <div style="text-align:center; margin-bottom:10px; font-size:1.2em;">
        <h2>Sample <span id="current-index">1</span> / <span id="total-count">0</span></h2>
    </div>

    <div class="display-area">
        <div class="full-row">
            <div class="full-label">Full HD Context (t)</div>
            <div class="full-box">
                <img id="img-full" src="">
            </div>
        </div>

        <div class="crops-row">
            <div class="crop-box"><div class="crop-label">t - 30</div><img id="img-m30" src=""></div>
            <div class="crop-box"><div class="crop-label">t - 20</div><img id="img-m20" src=""></div>
            <div class="crop-box"><div class="crop-label">t - 10</div><img id="img-m10" src=""></div>
            <div class="crop-box target-box"><div class="crop-label target-label">Target (t)</div><img id="img-tgt" src=""></div>
        </div>
    </div>

    <div class="controls">
        <div class="btn-group">
            <button class="btn-choice" onclick="selectChoice('Ascending')" id="btn-asc">1. 계단을 올라가거나, 한 쪽 발(foot)이 보이며 계단 쪽으로 향하는 경우<br>(Ascending, 사람에 가린 경우 가정하여 판단)</button>
            <button class="btn-choice" onclick="selectChoice('Descending')" id="btn-desc">2. 계단을 내려오거나, 승강장에서 출입문을 향해 이동하는 경우<br>(Descending)</button>
            <button class="btn-choice" onclick="selectChoice('Passing')" id="btn-pass">3. 그 외 (판단 불가 경우 포함)<br>(Passing)</button>
        </div>

        <div style="margin-top:10px;">
            <button class="btn-nav" onclick="prevSample()">← 이전 (Left)</button>
            <button class="btn-nav" onclick="nextSample()">다음 (Right) →</button>
        </div>

        <div id="finish-area" class="finish-area">
            <div class="finish-msg">🎉 모든 설문이 완료되었습니다!</div>
            <button class="btn-save" onclick="downloadCSV()">결과 저장 (Download CSV)</button>
        </div>
        <div style="color:#7f8c8d; margin-top:10px;">단축키: [1], [2], [3] 선택 &nbsp;|&nbsp; [←], [→] 이동</div>
    </div>
</div>

<script>
    const samples = {samples_json};
    let currentIndex = 0;
    const answers = {{}}; 

    document.getElementById('total-count').innerText = samples.length;
    loadSample(0);

    function loadSample(index) {{
        if (index < 0 || index >= samples.length) return;
        currentIndex = index;
        const s = samples[index];

        document.getElementById('current-index').innerText = index + 1;
        document.getElementById('img-m30').src = s.m30;
        document.getElementById('img-m20').src = s.m20;
        document.getElementById('img-m10').src = s.m10;
        document.getElementById('img-tgt').src = s.tgt;
        document.getElementById('img-full').src = s.full;

        updateButtons();
        checkCompletion();
    }}

    function selectChoice(c) {{
        answers[samples[currentIndex].id] = c;
        updateButtons();
        if (currentIndex < samples.length - 1) {{
            setTimeout(() => loadSample(currentIndex + 1), 150);
        }} else {{
            checkCompletion();
        }}
    }}

    function checkCompletion() {{
        if (Object.keys(answers).length === samples.length) {{
            document.getElementById('finish-area').style.display = 'block';
            document.getElementById('finish-area').scrollIntoView({{behavior: "smooth"}});
        }}
    }}

    function updateButtons() {{
        const ans = answers[samples[currentIndex].id];
        const mapping = {{'Ascending': 'asc', 'Descending': 'desc', 'Passing': 'pass'}};
        Object.keys(mapping).forEach(opt => {{
            const btn = document.getElementById('btn-' + mapping[opt]);
            if (opt === ans) btn.classList.add('selected');
            else btn.classList.remove('selected');
        }});
    }}

    function nextSample() {{ loadSample(currentIndex + 1); }}
    function prevSample() {{ loadSample(currentIndex - 1); }}

    function downloadCSV() {{
        let csv = "data:text/csv;charset=utf-8,Survey_ID,Selected_Class\\n";
        const sortedIds = Object.keys(answers).sort((a, b) => parseInt(a) - parseInt(b));
        sortedIds.forEach(id => csv += id + "," + answers[id] + "\\n");
        const link = document.createElement("a");
        link.href = encodeURI(csv);
        link.download = "survey_results_v6.csv";
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
    }}

    document.addEventListener('keydown', e => {{
        if(e.key==='1') selectChoice('Ascending');
        if(e.key==='2') selectChoice('Descending');
        if(e.key==='3') selectChoice('Passing');
        if(e.key==='ArrowRight') nextSample();
        if(e.key==='ArrowLeft') prevSample();
    }});
</script>
</body>
</html>
    """
    
    with open(HTML_OUTPUT, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"Success: Tool generated at {HTML_OUTPUT}")

if __name__ == "__main__":
    create_html_tool()

Success: Tool generated at survey_dataset_final_v6_extended/survey_tool.html
